# Fase 2 — EDA y limpieza

Dataset: **Almaty Air Quality History** (Kaggle, `fichka/almaty-air-quality-history`).

Objetivo de este notebook:
1. Cargar el CSV crudo y entender su esquema, nulos y rango temporal.
2. Limpiar las mediciones (drop nulos / negativos en PM2.5, parsear fechas).
3. Construir features temporales (hour, day, month, year, dayofweek).
4. Generar el target `aqi_class` con breakpoints EPA.
5. Imputar features ambientales (pm10, temperatura, humedad, um003) con la mediana.
6. Visualizar distribución de clases, evolución temporal y correlaciones.
7. Guardar `data/processed.csv` para la Fase 3 (modelado).

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

# Permitir importar src.preprocessing desde el notebook
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.preprocessing import (
    AQI_LABELS,
    add_aqi_class,
    clean_measurements,
    extract_temporal_features,
    impute_numeric_with_median,
    load_raw_data,
)

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 30)

DATA_DIR = PROJECT_ROOT / "data"
RAW_CSV = DATA_DIR / "air_quality_data.csv"
PROCESSED_CSV = DATA_DIR / "processed.csv"
print("Ruta raw:", RAW_CSV, "exists?", RAW_CSV.exists())

## 1. Carga e inspección inicial

In [ ]:
df_raw = load_raw_data(RAW_CSV)
print("Shape:", df_raw.shape)
df_raw.head()

In [ ]:
df_raw.info(memory_usage="deep")

In [ ]:
# Nulos por columna (absolutos y %)
nulls = pd.DataFrame({
    "nulls": df_raw.isna().sum(),
    "pct": (df_raw.isna().mean() * 100).round(2),
}).sort_values("pct", ascending=False)
nulls

In [ ]:
# Rango temporal del dataset
dt_series = pd.to_datetime(df_raw["datetime"], utc=True, errors="coerce")
print("Min datetime:", dt_series.min())
print("Max datetime:", dt_series.max())
print("Años cubiertos:", sorted(dt_series.dt.year.dropna().unique().tolist()))

In [ ]:
# Providers y estaciones únicas
print("Estaciones únicas (location_id):", df_raw["location_id"].nunique())
print("Providers:")
df_raw["provider_name"].value_counts(dropna=False)

In [ ]:
# Descriptivos de PM2.5 (target source)
df_raw["pm25"].describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.95, 0.99])

## 2. Limpieza + features temporales + target

In [ ]:
df = clean_measurements(df_raw)
print("Filas tras limpieza:", len(df), f"({len(df)/len(df_raw)*100:.1f}% del raw)")
df = extract_temporal_features(df)
df = add_aqi_class(df)
df.head()

In [ ]:
# Imputar columnas ambientales con la mediana
IMPUTE_COLS = ["pm10", "relativehumidity", "temperature", "um003"]
df, medians = impute_numeric_with_median(df, IMPUTE_COLS)
print("Medianas usadas para imputación:")
for k, v in medians.items():
    print(f"  {k}: {v:.3f}")
print("\nNulos restantes en columnas clave:")
df[IMPUTE_COLS + ["pm25", "aqi_class", "lat", "lon"]].isna().sum()

## 3. Visualizaciones

In [ ]:
# Distribución de clases AQI
class_counts = (
    df["aqi_class"].value_counts().sort_index().rename("count").to_frame()
)
class_counts["label"] = class_counts.index.map(AQI_LABELS)
class_counts["pct"] = (class_counts["count"] / class_counts["count"].sum() * 100).round(2)
class_counts

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
palette = ["#2ecc71", "#f1c40f", "#e67e22", "#e74c3c", "#8e44ad"]
sns.barplot(
    x=class_counts.index,
    y=class_counts["count"],
    palette=palette[: len(class_counts)],
    ax=ax,
)
ax.set_xticklabels([AQI_LABELS[i] for i in class_counts.index], rotation=20, ha="right")
ax.set_title("Distribución de clases de calidad del aire (Almaty)")
ax.set_xlabel("")
ax.set_ylabel("# mediciones")
plt.tight_layout()
plt.show()

In [ ]:
# Evolución temporal mensual de PM2.5
monthly = (
    df.set_index("datetime")["pm25"]
    .resample("MS")
    .agg(["mean", "median"])
    .dropna()
)
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(monthly.index, monthly["mean"], label="Media mensual", color="#e74c3c")
ax.plot(monthly.index, monthly["median"], label="Mediana mensual", color="#3498db")
ax.axhline(12, ls="--", color="#2ecc71", alpha=0.6, label="Límite Buena (12)")
ax.axhline(35.4, ls="--", color="#f39c12", alpha=0.6, label="Límite Moderada (35.4)")
ax.set_title("Evolución mensual de PM2.5 en Almaty")
ax.set_ylabel("PM2.5 (µg/m³)")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# PM2.5 promedio por hora del día (patrón diario)
hourly = df.groupby("hour")["pm25"].mean()
fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(x=hourly.index, y=hourly.values, color="#34495e", ax=ax)
ax.set_title("PM2.5 medio por hora del día")
ax.set_xlabel("Hora (UTC)")
ax.set_ylabel("PM2.5 medio (µg/m³)")
plt.tight_layout()
plt.show()

In [ ]:
# Correlación entre features numéricas
num_cols = [
    "pm25", "pm10", "relativehumidity", "temperature", "um003",
    "hour", "month", "dayofweek", "lat", "lon",
]
corr = df[num_cols].corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Matriz de correlación")
plt.tight_layout()
plt.show()

## 4. Guardar dataset procesado

In [ ]:
# Columnas finales que persistimos
FINAL_COLS = [
    "datetime", "location_id", "name", "provider_name", "lat", "lon",
    "pm25", "pm10", "relativehumidity", "temperature", "um003",
    "hour", "day", "month", "year", "dayofweek",
    "aqi_class", "aqi_label",
]
df_final = df[FINAL_COLS].copy()
df_final.to_csv(PROCESSED_CSV, index=False)
print("Guardado:", PROCESSED_CSV)
print("Shape final:", df_final.shape)
df_final.head()

## Notas para la Fase 3

- **Target**: `aqi_class` (0-4)
- **Features candidatas**: `pm10`, `relativehumidity`, `temperature`, `um003`, `hour`, `day`, `month`, `dayofweek`, `lat`, `lon`
- **NO usar como feature**: `pm25` (data leakage, el target se deriva de ahí).
- Las medianas de imputación quedan en la variable `medians` — se guardarán junto al modelo para que la API pueda imputar inputs del usuario que vengan vacíos.